# Hito 2: Métricas de fidelidad para transcripción MIDI

El objetivo de este notebook es definir una forma **cuantitativa y reproducible**
de evaluar la fidelidad de una transcripción musical automática.

Antes de comparar el pipeline

**Demucs → Basic Pitch**

con una variante que incorpore procesamiento de señal

**Demucs → DSP → Basic Pitch**,

construiremos un ejemplo controlado para comprender cómo se mide la coincidencia
entre una transcripción MIDI estimada y una referencia conocida.

En particular, estudiaremos tres conceptos:

- **Precision:** ¿qué proporción de las notas detectadas corresponde realmente a notas de la referencia?
- **Recall:** ¿qué proporción de las notas existentes en la referencia fue detectada?
- **F1-score:** ¿qué tan equilibradas están Precision y Recall?

Estas métricas permitirán reemplazar expresiones subjetivas como *“la transcripción
parece más limpia”* por una comparación objetiva entre distintas configuraciones
del pipeline.

## 1. ¿Qué significa acertar o equivocarse al transcribir una nota?

Para comparar una transcripción estimada con una referencia necesitamos clasificar
las notas detectadas en tres categorías:

- **True Positive (TP):** una nota de la referencia fue correctamente detectada.
- **False Positive (FP):** el sistema detectó una nota que no existe en la referencia.
  En este proyecto la interpretaremos como una **nota fantasma**.
- **False Negative (FN):** una nota presente en la referencia no fue detectada.
  La interpretaremos como una **nota perdida**.

A partir de estas cantidades se calculan:

$$
\mathrm{Precision} = \frac{TP}{TP + FP}
$$

$$
\mathrm{Recall} = \frac{TP}{TP + FN}
$$

$$
F_1 =
2\frac{\mathrm{Precision}\,\mathrm{Recall}}
{\mathrm{Precision}+\mathrm{Recall}}
$$

### Ejemplo controlado

Supongamos que la referencia contiene tres notas:

| Nota | Inicio (s) | Fin (s) |
|---|---:|---:|
| C4 | 0.00 | 0.50 |
| E4 | 1.00 | 1.50 |
| G4 | 2.00 | 2.50 |

y que el sistema estima:

| Nota | Inicio (s) | Fin (s) | Interpretación |
|---|---:|---:|---|
| C4 | 0.02 | 0.51 | detección correcta |
| E4 | 1.03 | 1.48 | detección correcta |
| F#4 | 1.70 | 1.90 | nota fantasma |

En este ejemplo:

- **TP = 2**
- **FP = 1**
- **FN = 1**, porque G4 no fue detectada.

Por tanto:

$$
\mathrm{Precision} = \frac{2}{3} \approx 0.667
$$

$$
\mathrm{Recall} = \frac{2}{3} \approx 0.667
$$

$$
F_1 \approx 0.667
$$

Este ejemplo será nuestra primera prueba de control antes de evaluar transcripciones
producidas por Basic Pitch.

In [1]:
# Ejemplo controlado: conteos definidos a partir del caso anterior

TP = 2  # C4 y E4 correctamente detectadas
FP = 1  # F#4: nota fantasma
FN = 1  # G4: nota perdida

precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)

print(f"TP = {TP}")
print(f"FP = {FP}")
print(f"FN = {FN}")
print()
print(f"Precision = {precision:.3f}")
print(f"Recall    = {recall:.3f}")
print(f"F1-score  = {f1:.3f}")

TP = 2
FP = 1
FN = 1

Precision = 0.667
Recall    = 0.667
F1-score  = 0.667


## 2. Representación de la referencia y la estimación

Hasta ahora hemos definido manualmente cuántos verdaderos positivos, falsos
positivos y falsos negativos existen.

El siguiente paso consiste en representar explícitamente las notas de la
**referencia** y de la **transcripción estimada**.

Cada nota se describirá mediante:

- su nombre musical;
- su número MIDI;
- el instante de inicio (*onset*);
- el instante de término (*offset*).

Posteriormente utilizaremos estas representaciones para decidir automáticamente
qué notas pueden considerarse coincidencias.

In [2]:
import pandas as pd

# Notas de referencia
reference_notes = pd.DataFrame(
    {
        "note": ["C4", "E4", "G4"],
        "midi": [60, 64, 67],
        "onset_s": [0.00, 1.00, 2.00],
        "offset_s": [0.50, 1.50, 2.50],
    }
)

# Notas estimadas por el sistema
estimated_notes = pd.DataFrame(
    {
        "note": ["C4", "E4", "F#4"],
        "midi": [60, 64, 66],
        "onset_s": [0.02, 1.03, 1.70],
        "offset_s": [0.51, 1.48, 1.90],
    }
)

print("Referencia")
display(reference_notes)

print("Estimación")
display(estimated_notes)

Referencia


,note,midi,onset_s,offset_s
0,C4,60,0.0,0.5
1,E4,64,1.0,1.5
2,G4,67,2.0,2.5


Estimación


,note,midi,onset_s,offset_s
0,C4,60,0.02,0.51
1,E4,64,1.03,1.48
2,F#4,66,1.70,1.90


## 3. Coincidencia entre notas: tolerancia temporal

En una transcripción automática no es razonable exigir que el inicio estimado de
una nota coincida exactamente con el inicio de la referencia.

Por ejemplo, una nota cuyo onset real ocurre en 1.000 s podría ser estimada en
1.030 s y seguir considerándose una detección correcta.

En esta primera aproximación consideraremos que una nota coincide con una
referencia cuando:

1. tiene el mismo número MIDI, y
2. la diferencia absoluta entre sus onsets no supera **50 ms**.

Además, cada nota estimada podrá asociarse como máximo a una nota de referencia.

In [3]:
# Tolerancia temporal para el onset
onset_tolerance_s = 0.050  # 50 ms

matches = []
matched_estimated = set()

for ref_idx, ref in reference_notes.iterrows():

    # Candidatas con el mismo pitch MIDI y que aún no han sido utilizadas
    candidates = estimated_notes[
        (estimated_notes["midi"] == ref["midi"])
        & (~estimated_notes.index.isin(matched_estimated))
    ].copy()

    if len(candidates) == 0:
        continue

    # Diferencia temporal respecto del onset de referencia
    candidates["onset_error_s"] = (
        candidates["onset_s"] - ref["onset_s"]
    ).abs()

    # Conservamos únicamente las candidatas dentro de la tolerancia
    candidates = candidates[
        candidates["onset_error_s"] <= onset_tolerance_s
    ]

    if len(candidates) == 0:
        continue

    # Si hubiera más de una candidata, elegimos la más cercana temporalmente
    est_idx = candidates["onset_error_s"].idxmin()
    est = estimated_notes.loc[est_idx]

    matches.append(
        {
            "reference_note": ref["note"],
            "estimated_note": est["note"],
            "reference_onset_s": ref["onset_s"],
            "estimated_onset_s": est["onset_s"],
            "onset_error_ms": 1000 * abs(est["onset_s"] - ref["onset_s"]),
        }
    )

    matched_estimated.add(est_idx)

matches_df = pd.DataFrame(matches)

display(matches_df)

,reference_note,estimated_note,reference_onset_s,estimated_onset_s,onset_error_ms
0,C4,C4,0.0,0.02,20.0
1,E4,E4,1.0,1.03,30.0


## 4. Cálculo automático de Precision, Recall y F1

Una vez identificadas las coincidencias entre referencia y estimación, podemos
derivar automáticamente los conteos utilizados por las métricas:

- **TP:** número de coincidencias encontradas.
- **FP:** notas estimadas que no pudieron asociarse a una referencia.
- **FN:** notas de referencia que no fueron detectadas.

De esta forma, las métricas dejan de depender de conteos introducidos manualmente
y pasan a ser el resultado del procedimiento de evaluación.

In [4]:
# Conteos derivados automáticamente del matching

TP_auto = len(matches_df)
FP_auto = len(estimated_notes) - TP_auto
FN_auto = len(reference_notes) - TP_auto

precision_auto = TP_auto / (TP_auto + FP_auto)
recall_auto = TP_auto / (TP_auto + FN_auto)

if precision_auto + recall_auto > 0:
    f1_auto = (
        2 * precision_auto * recall_auto
        / (precision_auto + recall_auto)
    )
else:
    f1_auto = 0.0

print(f"TP = {TP_auto}")
print(f"FP = {FP_auto}")
print(f"FN = {FN_auto}")
print()
print(f"Precision = {precision_auto:.3f}")
print(f"Recall    = {recall_auto:.3f}")
print(f"F1-score  = {f1_auto:.3f}")


TP = 2
FP = 1
FN = 1

Precision = 0.667
Recall    = 0.667
F1-score  = 0.667


## 5. Evaluación estándar con `mir_eval`

El procedimiento anterior nos permitió comprender explícitamente cómo se construyen
las coincidencias entre notas y cómo se obtienen Precision, Recall y F1.

Ahora compararemos ese resultado con `mir_eval`, una librería especializada en
evaluación de sistemas de Music Information Retrieval.

Evaluaremos dos criterios:

1. **Nota sin considerar offset:** pitch + onset.
2. **Nota considerando offset:** pitch + onset + duración aproximada.

Esta distinción será útil posteriormente para estudiar si el procesamiento DSP
afecta de manera diferente la detección del ataque y la duración de las notas.

In [5]:
import numpy as np
import mir_eval

# Convertimos los DataFrames al formato requerido por mir_eval:
# intervalos [onset, offset] y pitches expresados en Hz.

ref_intervals = reference_notes[
    ["onset_s", "offset_s"]
].to_numpy(dtype=float)

est_intervals = estimated_notes[
    ["onset_s", "offset_s"]
].to_numpy(dtype=float)


def midi_to_hz(midi_notes):
    """Convierte números MIDI a frecuencia fundamental en Hz."""
    midi_notes = np.asarray(midi_notes, dtype=float)
    return 440.0 * 2 ** ((midi_notes - 69.0) / 12.0)


ref_pitches_hz = midi_to_hz(reference_notes["midi"])
est_pitches_hz = midi_to_hz(estimated_notes["midi"])

print("Pitches de referencia (Hz):")
print(np.round(ref_pitches_hz, 2))

print("\nPitches estimados (Hz):")
print(np.round(est_pitches_hz, 2))

Pitches de referencia (Hz):
[261.63 329.63 392.  ]

Pitches estimados (Hz):
[261.63 329.63 369.99]


In [6]:
# 1. Evaluación de nota: pitch + onset
#    Ignoramos el offset para medir principalmente detección de nota e inicio.

p_no_offset, r_no_offset, f1_no_offset, overlap_no_offset = (
    mir_eval.transcription.precision_recall_f1_overlap(
        ref_intervals,
        ref_pitches_hz,
        est_intervals,
        est_pitches_hz,
        onset_tolerance=0.050,
        pitch_tolerance=50.0,
        offset_ratio=None,
    )
)

# 2. Evaluación más exigente: pitch + onset + offset
#    Aquí también exigimos que la duración de la nota sea razonablemente correcta.

p_with_offset, r_with_offset, f1_with_offset, overlap_with_offset = (
    mir_eval.transcription.precision_recall_f1_overlap(
        ref_intervals,
        ref_pitches_hz,
        est_intervals,
        est_pitches_hz,
        onset_tolerance=0.050,
        pitch_tolerance=50.0,
        offset_ratio=0.2,
        offset_min_tolerance=0.050,
    )
)

print("=== Pitch + onset ===")
print(f"Precision = {p_no_offset:.3f}")
print(f"Recall    = {r_no_offset:.3f}")
print(f"F1-score  = {f1_no_offset:.3f}")

print("\n=== Pitch + onset + offset ===")
print(f"Precision = {p_with_offset:.3f}")
print(f"Recall    = {r_with_offset:.3f}")
print(f"F1-score  = {f1_with_offset:.3f}")

=== Pitch + onset ===
Precision = 0.667
Recall    = 0.667
F1-score  = 0.667

=== Pitch + onset + offset ===
Precision = 0.667
Recall    = 0.667
F1-score  = 0.667


## 6. Experimento de control: error en la duración de una nota

Hasta ahora, las notas correctamente detectadas también presentan offsets cercanos
a los de la referencia. Por eso las métricas con y sin evaluación de offset
producen el mismo resultado.

Para comprobar que ambas métricas capturan aspectos diferentes de la
transcripción, modificaremos deliberadamente **solo el offset de E4**.

Mantendremos:

- el mismo pitch;
- el mismo onset estimado;

pero haremos que E4 termine mucho antes de lo que indica la referencia.

Esperamos que:

- la evaluación **pitch + onset** no cambie;
- la evaluación **pitch + onset + offset** penalice esta nota.

In [7]:
# Creamos una copia para no modificar el ejemplo original
estimated_notes_bad_offset = estimated_notes.copy()

# E4 corresponde a la segunda fila.
# Su offset original estimado era 1.48 s.
# Lo desplazamos deliberadamente a 1.20 s.
estimated_notes_bad_offset.loc[
    estimated_notes_bad_offset["note"] == "E4",
    "offset_s"
] = 1.20

display(estimated_notes_bad_offset)

,note,midi,onset_s,offset_s
0,C4,60,0.02,0.51
1,E4,64,1.03,1.20
2,F#4,66,1.70,1.90


In [8]:
# Nuevos intervalos con el offset de E4 alterado
est_intervals_bad_offset = estimated_notes_bad_offset[
    ["onset_s", "offset_s"]
].to_numpy(dtype=float)

# Pitch + onset: ignoramos la duración
p_bad_no_offset, r_bad_no_offset, f1_bad_no_offset, _ = (
    mir_eval.transcription.precision_recall_f1_overlap(
        ref_intervals,
        ref_pitches_hz,
        est_intervals_bad_offset,
        est_pitches_hz,
        onset_tolerance=0.050,
        pitch_tolerance=50.0,
        offset_ratio=None,
    )
)

# Pitch + onset + offset: también evaluamos la duración
p_bad_with_offset, r_bad_with_offset, f1_bad_with_offset, _ = (
    mir_eval.transcription.precision_recall_f1_overlap(
        ref_intervals,
        ref_pitches_hz,
        est_intervals_bad_offset,
        est_pitches_hz,
        onset_tolerance=0.050,
        pitch_tolerance=50.0,
        offset_ratio=0.2,
        offset_min_tolerance=0.050,
    )
)

print("=== Pitch + onset ===")
print(f"Precision = {p_bad_no_offset:.3f}")
print(f"Recall    = {r_bad_no_offset:.3f}")
print(f"F1-score  = {f1_bad_no_offset:.3f}")

print("\n=== Pitch + onset + offset ===")
print(f"Precision = {p_bad_with_offset:.3f}")
print(f"Recall    = {r_bad_with_offset:.3f}")
print(f"F1-score  = {f1_bad_with_offset:.3f}")

=== Pitch + onset ===
Precision = 0.667
Recall    = 0.667
F1-score  = 0.667

=== Pitch + onset + offset ===
Precision = 0.333
Recall    = 0.333
F1-score  = 0.333


### Interpretación del experimento

El cambio introducido en el offset de E4 no modificó su pitch ni su onset.
Por esta razón, la métrica basada en **pitch + onset** permaneció sin cambios:

$$
F_{1,\mathrm{onset}} = 0.667
$$

Sin embargo, al exigir también una duración razonablemente correcta,
E4 dejó de considerarse una coincidencia y la métrica disminuyó a:

$$
F_{1,\mathrm{onset+offset}} = 0.333
$$

Este resultado muestra que la fidelidad de una transcripción MIDI no tiene
necesariamente una única dimensión.

En el experimento posterior con el pipeline de audio será conveniente conservar,
como mínimo, dos métricas:

- **F1 de nota (pitch + onset):** capacidad para identificar correctamente
  las notas y sus instantes de inicio.
- **F1 de nota con offset (pitch + onset + offset):** capacidad para representar
  además razonablemente su duración.

Esto permitirá evaluar si una etapa de DSP mejora la detección de notas sin
asumir que necesariamente mejora también su duración.

## 7. Función reutilizable para evaluar una transcripción

Los experimentos anteriores permitieron comprender paso a paso cómo se construyen
las métricas de evaluación.

A partir de este punto encapsularemos el procedimiento en una función reutilizable.
La función recibirá dos tablas de notas:

- una **referencia**;
- una **estimación**;

y devolverá Precision, Recall y F1 para dos criterios:

1. **pitch + onset**;
2. **pitch + onset + offset**.

Esta función será utilizada posteriormente para comparar de manera consistente
las distintas configuraciones del pipeline.

In [9]:
def evaluate_transcription(
    reference,
    estimated,
    onset_tolerance=0.050,
    pitch_tolerance=50.0,
    offset_ratio=0.2,
    offset_min_tolerance=0.050,
):
    """
    Evalúa una transcripción estimada respecto de una referencia.

    Parameters
    ----------
    reference : pandas.DataFrame
        Debe contener las columnas:
        midi, onset_s, offset_s.

    estimated : pandas.DataFrame
        Debe contener las columnas:
        midi, onset_s, offset_s.

    onset_tolerance : float
        Tolerancia para el onset, en segundos.

    pitch_tolerance : float
        Tolerancia de pitch, en cents.

    offset_ratio : float
        Tolerancia relativa utilizada para evaluar offsets.

    offset_min_tolerance : float
        Tolerancia mínima para offsets, en segundos.

    Returns
    -------
    pandas.DataFrame
        Tabla con Precision, Recall y F1 para:
        - pitch + onset
        - pitch + onset + offset
    """

    # Intervalos [onset, offset]
    ref_intervals = reference[
        ["onset_s", "offset_s"]
    ].to_numpy(dtype=float)

    est_intervals = estimated[
        ["onset_s", "offset_s"]
    ].to_numpy(dtype=float)

    # Conversión MIDI → Hz
    ref_pitches_hz = midi_to_hz(reference["midi"])
    est_pitches_hz = midi_to_hz(estimated["midi"])

    # Evaluación 1: pitch + onset
    p_onset, r_onset, f1_onset, _ = (
        mir_eval.transcription.precision_recall_f1_overlap(
            ref_intervals,
            ref_pitches_hz,
            est_intervals,
            est_pitches_hz,
            onset_tolerance=onset_tolerance,
            pitch_tolerance=pitch_tolerance,
            offset_ratio=None,
        )
    )

    # Evaluación 2: pitch + onset + offset
    p_offset, r_offset, f1_offset, _ = (
        mir_eval.transcription.precision_recall_f1_overlap(
            ref_intervals,
            ref_pitches_hz,
            est_intervals,
            est_pitches_hz,
            onset_tolerance=onset_tolerance,
            pitch_tolerance=pitch_tolerance,
            offset_ratio=offset_ratio,
            offset_min_tolerance=offset_min_tolerance,
        )
    )

    results = pd.DataFrame(
        {
            "criterion": [
                "pitch + onset",
                "pitch + onset + offset",
            ],
            "precision": [
                p_onset,
                p_offset,
            ],
            "recall": [
                r_onset,
                r_offset,
            ],
            "f1": [
                f1_onset,
                f1_offset,
            ],
        }
    )

    return results

In [10]:
results_original = evaluate_transcription(
    reference_notes,
    estimated_notes,
)

results_original

,criterion,precision,recall,f1
0,pitch + onset,0.666667,0.666667,0.666667
1,pitch + onset + offset,0.666667,0.666667,0.666667


In [11]:
results_bad_offset = evaluate_transcription(
    reference_notes,
    estimated_notes_bad_offset,
)

results_bad_offset

,criterion,precision,recall,f1
0,pitch + onset,0.666667,0.666667,0.666667
1,pitch + onset + offset,0.333333,0.333333,0.333333


## 8. Conclusión del Hito 2

En este notebook se construyó y verificó un procedimiento reproducible para
evaluar la fidelidad de una transcripción musical automática.

El desarrollo siguió tres niveles de verificación:

1. cálculo manual de Precision, Recall y F1;
2. construcción explícita de coincidencias entre notas;
3. comparación con la implementación estándar de `mir_eval`.

Los tres procedimientos produjeron resultados consistentes en el ejemplo
controlado.

Además, se comprobó experimentalmente que dos dimensiones de la transcripción
pueden comportarse de manera diferente:

- **F1 de nota (pitch + onset)**;
- **F1 de nota con offset (pitch + onset + offset)**.

Por esta razón, el efecto de una etapa de DSP no se interpretará simplemente
como “mejor” o “peor”. Se evaluará si modifica la detección de notas, su duración,
o ambas.

### Siguiente etapa

Para aplicar estas métricas al pipeline real se necesita una referencia
independiente (*ground truth*) que permita comparar:

**MIDI de referencia ↔ MIDI estimado por Basic Pitch**

Una vez definida esa referencia será posible contrastar, bajo las mismas
condiciones:

**Demucs → Basic Pitch**

frente a:

**Demucs → DSP → Basic Pitch**.